In [38]:
from pathlib import Path

import numpy as np
from skmultilearn.model_selection import iterative_train_test_split

In [39]:
MSGO_CLASSES = {
    "Plane": 0,
    "Bridge": 1,
    "Intersection": 2,
    "Roundabout": 3,
    "Vehicle": 4,
    "Ship": 5,
}

NUM_CLASSES = len(MSGO_CLASSES)

In [50]:
def build_image_to_counts(root_dir: str) -> dict[str, dict[int, int]]:
    root_path = Path(root_dir)
    image_to_counts = {}

    for split_path in root_path.iterdir():
        if not split_path.is_dir() or split_path.is_file():
            continue

        images_dir = split_path / "images"
        labels_dir = split_path / "labels"

        for label_file in labels_dir.glob("*.txt"):
            img_file = images_dir / f"{label_file.stem}.jpg"

            counts = {}
            with open(label_file) as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue
                class_id = int(parts[0])
                counts[class_id] = counts.get(class_id, 0) + 1

            image_to_counts[str(img_file)] = counts

    return image_to_counts

In [51]:
image_to_counts = build_image_to_counts("D:\\stuff\\datasets\\MSGOv1")

In [52]:
image_paths = list(image_to_counts.keys())
num_images = len(image_paths)

y_counts = np.zeros((num_images, NUM_CLASSES), dtype=int)
for i, path in enumerate(image_paths):
    counts = image_to_counts[path]
    for class_id, count in counts.items():
        y_counts[i, class_id] = count

X = np.array(image_paths).reshape(-1, 1)

In [59]:
X_train, y_train_counts, X_temp, y_temp_counts = iterative_train_test_split(
    X,
    y_counts,
    test_size=0.2,  # 20% -> val + test
)

X_val, y_val_counts, X_test, y_test_counts = iterative_train_test_split(X_temp, y_temp_counts, test_size=0.5)

X_train_paths = X_train.flatten().tolist()
X_val_paths = X_val.flatten().tolist()
X_test_paths = X_test.flatten().tolist()

print(f"Total images: {len(X)}")
print(f"Train images: {len(X_train_paths)} ({len(X_train_paths) / len(X):.1%})")
print(f"Validation images: {len(X_val_paths)} ({len(X_val_paths) / len(X):.1%})")
print(f"Test images: {len(X_test_paths)} ({len(X_test_paths) / len(X):.1%})")

Total images: 39175
Train images: 31340 (80.0%)
Validation images: 3918 (10.0%)
Test images: 3917 (10.0%)


In [58]:
def check_distribution(paths, image_to_counts_map, num_classes):
    total_counts = np.zeros(num_classes, dtype=int)
    for path in paths:
        counts = image_to_counts_map.get(path, {})
        for class_id, count in counts.items():
            total_counts[class_id] += count
    return total_counts


train_counts = check_distribution(X_train_paths, image_to_counts, NUM_CLASSES)
val_counts = check_distribution(X_val_paths, image_to_counts, NUM_CLASSES)
test_counts = check_distribution(X_test_paths, image_to_counts, NUM_CLASSES)
total_counts = train_counts + val_counts + test_counts

print(f"Class Names: {list(MSGO_CLASSES.keys())}")
print(f"Total Instances: {total_counts}")
print(f"Train Instances: {train_counts} ({(train_counts / total_counts * 100).round(1)}%)")
print(f"Val Instances:   {val_counts} ({(val_counts / total_counts * 100).round(1)}%)")
print(f"Test Instances:  {test_counts} ({(test_counts / total_counts * 100).round(1)}%)")

Class Names: ['Plane', 'Bridge', 'Intersection', 'Roundabout', 'Vehicle', 'Ship']
Total Instances: [ 63950   8620  10156   1718 750564 174630]
Train Instances: [ 52348   6863   8200   1318 673488 127644] ([81.9 79.6 80.7 76.7 89.7 73.1]%)
Val Instances:   [ 5735   837   982   198 36842 21950] ([ 9.   9.7  9.7 11.5  4.9 12.6]%)
Test Instances:  [ 5867   920   974   202 40234 25036] ([ 9.2 10.7  9.6 11.8  5.4 14.3]%)
